# Maxwell photonic device

The geometry, Q2 local fields and P1 trace dimensions follow Lanteri et al. (2018), Section 6.3. The incident transient is explicitly defined in the case documentation. A classical central-DG sequence provides a numerical baseline, with separate space, time and quadrature controls. It is not an exact solution.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/d4b810786d85a7cc65010b2b978980c8cbdcebc016956bd998949f51342089dd/66_maxwell_nanoguide-companion.zip"
COMPANION_SHA256 = "d4b810786d85a7cc65010b2b978980c8cbdcebc016956bd998949f51342089dd"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("waves/maxwell/66_maxwell_nanoguide.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.meshes.cartesian import CartesianMacroMesh
data = root / "examples/results/maxwell-nanoguide"

## Small operator check

A constant magnetic field in a PEC cavity has zero curl. Its electric field remains zero and the physical field is stationary. This checks the Cartesian path cheaply; the photonic comparison uses the independently acquired full fields.

## User-defined time-discrete equations

The primary loop uses the shared free equation builders displayed below.
`LocalEquations` declares each local midpoint field and its signed trace maps;
`Equation` supplies the exterior impedance balance. The magnetic update has
zero trace coordinates and uses the same global assembly/reconstruction core.
With $C=-\mathrm{curl}$, the electric stage and global face equation are

$$
\begin{aligned}
 M_e w+\tfrac{\tau}{2}Q\lambda
 &=M_e E_{\mathrm{old}}+\tfrac{\tau}{2}(F-C^\top H),\\
 -\sum_K Q_K^\top w_K+Z\lambda&=-b,\\
 E_{\mathrm{new}}&=2w-E_{\mathrm{old}}.
\end{aligned}
$$

The initialization takes a half electric kick after the physical mass and PEC
projections. Each later step first solves

$$
 M_h H_{\mathrm{new}}=M_h H_{\mathrm{old}}+\Delta t\,C E.
$$

The helper accepts explicit material/boundary callbacks, face partitions and
execution settings. A physical solver name does not select these equations.

In [ ]:
import inspect
from IPython.display import Code
from examples.tutorial_maxwell_equations import (
    prepare, initialize, advance, l2_errors, compare_state,
    electric_equations, magnetic_equations, electric_global,
)

# These are the exact source forms consumed by the generic core below.
for builder in (electric_equations, magnetic_equations, electric_global):
    display(Code(inspect.getsource(builder), language="python"))

In [ ]:
with threadpool_limits(1):
    mesh = CartesianMacroMesh(2)
    operator = prepare(
        mesh, time_step=0.005, degree=2, local_refinement=2,
        absorbing=None, quadrature_order=6,
    )
    first = initialize(operator, 0.0, np.array([0.2, -0.3]))
    trajectory = [first]
    last = first
    for _ in range(8):
        last = advance(operator, last)
        trajectory.append(last)
    assert abs(last.energy - first.energy) < 1e-12
    assert max(np.max(np.abs(e)) for e in last.electric) < 1e-12
print("Cartesian Q2 stationary-field check passed with user-defined equations")

## Compatibility and physical callback controls

The same Cartesian stationary field is compared with the earlier stepper.
A separate short control uses the device's actual permittivity and incident
boundary callbacks, with Q2 fields and explicitly split P1 traces. This small
control checks equation/option wiring; the full photonic figures below come
from their independently acquired refinement sequences.

In [ ]:
with threadpool_limits(1):
    references = [initialize(operator, 0.0, np.array([0.2, -0.3]), original=True)]
    for _ in range(8):
        references.append(advance(operator, references[-1], original=True))
    assert abs(references[-1].energy - references[0].energy) < 1e-12
    assert max(np.max(np.abs(e)) for e in references[-1].electric) < 1e-12
for own, reference in zip(trajectory, references, strict=True):
    assert max(compare_state(own, reference).values()) < 2e-11

In [ ]:
from examples.maxwell_nanoguide import NanoWaveguide
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.vector.curl import TangentialTraceSpace

model = NanoWaveguide()
callback_mesh = CartesianMacroMesh(2, 2, (0.0, 10.0, 0.0, 10.0))
trace = TangentialTraceSpace(SkeletonSpace(
    callback_mesh,
    tuple(FaceSpace.uniform(1, 2) for _ in callback_mesh.faces),
))
with threadpool_limits(1):
    callback_operator = prepare(
        callback_mesh, time_step=0.005, degree=2, local_refinement=2,
        skeleton=trace, permittivity=model.permittivity,
        absorbing=1.0, boundary_data=model.boundary, quadrature_order=6,
    )
    current = initialize(callback_operator, 0.0, 0.0)
    snapshots = [current]
    for _ in range(2):
        current = advance(callback_operator, current)
        snapshots.append(current)
    callback_references = [initialize(callback_operator, 0.0, 0.0, original=True)]
    for _ in range(2):
        callback_references.append(advance(callback_operator, callback_references[-1], original=True))
for own, reference in zip(snapshots, callback_references, strict=True):
    assert max(compare_state(own, reference).values()) < 2e-11
print({
    "electric_time": current.electric_time,
    "magnetic_time": current.magnetic_time,
    "tangential_moment_norm": current.constraint_moment_norm,
    "energy_balance_residual": current.energy_balance_residual,
})

In [ ]:
if HISTORICAL_REPLAY:
    import hashlib
    record = json.loads((data / "comparison.json").read_text())
    controls = json.loads((data / "refinement-controls.json").read_text())
    verified = set()
    for row in [*record["comparisons"], *controls.values()]:
        for key in ("field", "reference"):
            metadata = row[key]
            if metadata["file"] not in verified:
                assert hashlib.sha256((data / metadata["file"]).read_bytes()).hexdigest() == metadata["sha256"]
                verified.add(metadata["file"])
    for row in record["comparisons"]:
        print(row["field"]["file"], "relative component L2:", row["relative_l2"], "combined:", row["combined_relative_l2"])
    for name, row in controls.items():
        print(name, "combined relative increment:", row["combined_relative_l2"])
    print("All field digests verified")
else:
    print("Original external archive replay is optional; use --historical with its attributed payloads.")


In [ ]:
if HISTORICAL_REPLAY:
    for name in ["material", "components", "errors", "profiles", "published-electric", "published-coarse"]:
        display(Image(filename=str(root / "docs/figures/maxwell-nanoguide" / (name + ".png"))))
else:
    print("Original external archive replay is optional; use --historical with its attributed payloads.")


Spatial fields use the same component scales. Macro boundaries remain visible; profiles retain independent fine-cell traces. Printed article panels are attributed to Figures 6.9–6.10. Detailed phase/amplitude agreement requires the same initial and incident-wave conventions, which the article does not fully specify.

The current main assembles the nanoguide material and driven absorbing boundary through public electric/magnetic equations. Its two steps are a callback and original-block comparison; they do not reproduce the final-time published device image. The complete PyMHM device acquisition is `python -m examples.maxwell_nanoguide`. The retained independent central-DG acquisition belongs to an external instrumented driver whose source and coefficient payloads are not distributed; the original comparison above is therefore explicitly optional.
